Building a Context-Aware AI Chatbot using NLP Techniques.

In [262]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import os
import re
import sqlite3
from datetime import datetime

import pandas as pd
from transformers import pipeline

import speech_recognition as sr
import pyttsx3

print("Libraries imported successfully.")

Libraries imported successfully.


In [263]:
# ============================================================
# CELL 2: LOAD DATASET
# ============================================================

DATA_PATH = "./dataset/customers.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

Dataset loaded successfully.
Shape: (1000, 2)


,user_input,intent
0,"Cancel my subscription, please.",cancellation
1,I forgot my password.,password_reset
2,Can you help me with my account?,account_help
3,I want to return my order.,return_request
4,What time do you open?,business_hours


In [264]:
# ============================================================
# CELL 3: BASIC DATA CLEANING AND EXPLORATION
# ============================================================

def clean_dataset(data):
    """
    Clean the customer support intent dataset.
    Duplicate rows are not removed because repeated examples
    are useful in intent datasets.
    """
    data = data.copy()

    data.columns = data.columns.str.strip().str.lower()

    data = data.dropna(subset=["user_input", "intent"])

    data["user_input"] = data["user_input"].astype(str).str.strip()
    data["intent"] = data["intent"].astype(str).str.strip()

    data = data[data["user_input"] != ""]
    data = data[data["intent"] != ""]

    return data


df = clean_dataset(df)

print("Cleaned dataset shape:", df.shape)
print("\nIntent distribution:")
print(df["intent"].value_counts())

Cleaned dataset shape: (1000, 2)

Intent distribution:
intent
business_hours       219
payment_update       117
account_help         103
order_status         102
technical_support     98
cancellation          96
service_info          91
password_reset        90
return_request        84
Name: count, dtype: int64


In [265]:
# ============================================================
# CELL 4: INTENTS AND RESPONSE SYSTEM
# ============================================================

candidate_labels = sorted(df["intent"].unique().tolist())

responses = {
    "greeting": "Hello! How can I help you today?",
    "small_talk": "You're welcome. Let me know if you need anything else.",

    "account_help": "Sure, I can help with your account. Please tell me what issue you are facing.",
    "business_hours": "Our support team is available from 9 AM to 6 PM, Monday to Saturday.",
    "cancellation": "I can help you cancel your subscription. Please confirm if you want to continue.",
    "order_status": "I can help you track your order. Please provide your order ID or tracking number.",
    "password_reset": "You can reset your password using the 'Forgot Password' option. I can guide you step by step.",
    "payment_update": "You can update your payment method from your account billing or payment settings.",
    "return_request": "I can help you start a return request. Please share your order number.",
    "service_info": "We provide account support, order tracking, returns, payment help, cancellation support, and technical assistance.",
    "technical_support": "I can help with technical support. Please describe the issue you are facing."
}

print("Candidate labels:")
print(candidate_labels)

Candidate labels:
['account_help', 'business_hours', 'cancellation', 'order_status', 'password_reset', 'payment_update', 'return_request', 'service_info', 'technical_support']


In [266]:
# ============================================================
# CELL 5: LOAD PRETRAINED TRANSFORMER MODEL
# ============================================================

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

print("Pretrained transformer loaded successfully.")

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 62bc151f-18b8-48b0-85c3-416b3a537913)')' thrown while requesting HEAD https://huggingface.co/facebook/bart-large-mnli/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 9df7decd-9a33-43f5-aff6-d1bb9a6414cb)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-mnli/d7645e127eaf1aefc7862fd59a17a5aa8558b8ce/config.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 3928dc9f-af2a-4406-8dfd-d157506b938e)')' thrown while requesting HEAD https://huggingface.co/api/resolve

Pretrained transformer loaded successfully.


In [267]:
# ============================================================
# VOICE ENGINE
# ============================================================

tts_engine = pyttsx3.init()

tts_engine.setProperty("rate", 170)
tts_engine.setProperty("volume", 1.0)

In [268]:
# ============================================================
# SPEECH TO TEXT
# ============================================================

def listen_to_user():
    """
    Capture voice input from microphone.
    """

    recognizer = sr.Recognizer()

    with sr.Microphone() as source:

        st.info("🎤 Listening... Please speak.")

        recognizer.adjust_for_ambient_noise(source)

        audio = recognizer.listen(source)

    try:
        text = recognizer.recognize_google(audio)

        return text

    except sr.UnknownValueError:
        return None

    except sr.RequestError:
        return None

In [269]:
# ============================================================
# TEXT TO SPEECH
# ============================================================

def speak_response(text):
    """
    Convert chatbot text response into speech.
    """

    tts_engine.say(text)
    tts_engine.runAndWait()

In [270]:
# ============================================================
# CELL 6: RULE-BASED INTENT DETECTION
# ============================================================

def rule_based_intent(user_input):
    """
    Handles short, direct, and obvious user messages before using transformer.
    This improves chatbot reliability for simple queries.
    """

    text = user_input.lower().strip()

    greetings = ["hi", "hello", "hey", "salam", "assalamualaikum"]
    thanks = ["ok", "okay", "thanks", "thank you", "alright", "fine"]

    if text in greetings:
        return "greeting", 1.0

    if text in thanks:
        return "small_talk", 1.0

    if any(word in text for word in ["cancel", "cancellation", "unsubscribe"]):
        return "cancellation", 1.0

    if any(word in text for word in ["forgot password", "forget password", "reset password", "password"]):
        return "password_reset", 1.0

    if any(word in text for word in ["order", "package", "parcel", "tracking", "shipment"]):
        return "order_status", 1.0

    if any(word in text for word in ["return", "refund", "replace"]):
        return "return_request", 1.0

    if any(word in text for word in ["payment", "billing", "card", "pay"]):
        return "payment_update", 1.0

    if any(word in text for word in ["technical", "support", "error", "issue", "problem", "not working"]):
        return "technical_support", 1.0

    if any(word in text for word in ["service", "services", "offer", "provide"]):
        return "service_info", 1.0

    if any(word in text for word in ["open", "hours", "working hours", "time"]):
        return "business_hours", 1.0

    return None, 0.0

In [271]:
# ============================================================
# CELL 7: TRANSFORMER INTENT PREDICTION
# ============================================================

def predict_intent_transformer(user_input):
    """
    Predict user intent using pretrained zero-shot transformer.
    No model training is required.
    """

    result = zero_shot_classifier(
        user_input,
        candidate_labels=candidate_labels
    )

    intent = result["labels"][0]
    confidence = float(result["scores"][0])

    return intent, confidence

In [272]:
# ============================================================
# CELL 8: CONTEXT MEMORY
# ============================================================

context_memory = {}

def update_context(user_id, intent):
    """
    Store the user's latest important intent.
    """

    context_memory[user_id] = {
        "last_intent": intent,
        "updated_at": datetime.now().isoformat()
    }


def get_last_intent(user_id):
    """
    Retrieve the user's previous intent.
    """

    return context_memory.get(user_id, {}).get("last_intent")


def clear_context(user_id):
    """
    Clear saved context for a user.
    """

    if user_id in context_memory:
        del context_memory[user_id]

In [273]:
# ============================================================
# CELL 9: SQLITE DATABASE FOR CHAT HISTORY
# ============================================================

DB_PATH = "chat_history.db"

def init_database():
    """
    Create SQLite database table and safely update old schema.
    """

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id TEXT,
            user_message TEXT,
            bot_response TEXT,
            predicted_intent TEXT,
            confidence REAL,
            model_used TEXT,
            timestamp TEXT
        )
    """)

    cursor.execute("PRAGMA table_info(chat_history)")
    columns = [column[1] for column in cursor.fetchall()]

    if "model_used" not in columns:
        cursor.execute("ALTER TABLE chat_history ADD COLUMN model_used TEXT")

    conn.commit()
    conn.close()


def save_chat(user_id, user_message, bot_response, intent, confidence, model_used):
    """
    Save chat conversation to database.
    """

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO chat_history
        (
            user_id,
            user_message,
            bot_response,
            predicted_intent,
            confidence,
            model_used,
            timestamp
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        user_id,
        user_message,
        bot_response,
        intent,
        float(confidence),
        model_used,
        datetime.now().isoformat()
    ))

    conn.commit()
    conn.close()


def load_chat_history(limit=10):
    """
    Load recent chat history from database.
    """

    conn = sqlite3.connect(DB_PATH)

    query = """
        SELECT
            user_message,
            bot_response,
            predicted_intent,
            confidence,
            model_used,
            timestamp
        FROM chat_history
        ORDER BY id DESC
        LIMIT ?
    """

    history = pd.read_sql_query(query, conn, params=(limit,))
    conn.close()

    return history


init_database()

print("Database initialized successfully.")

Database initialized successfully.


In [274]:
# ============================================================
# CELL 10: MAIN CHATBOT ENGINE
# ============================================================

def chatbot_response(user_input, user_id="notebook_user"):
    """
    Main context-aware chatbot function.

    Flow:
    1. Check empty input
    2. Check context follow-up
    3. Apply rule-based detection
    4. Use pretrained transformer
    5. Generate response
    6. Update context
    7. Save chat history
    """

    if not user_input.strip():
        return {
            "response": "Please type your message.",
            "intent": "empty",
            "confidence": 0.0,
            "model_used": "Rule Based"
        }

    last_intent = get_last_intent(user_id)

    # Context follow-up for order ID
    if last_intent == "order_status" and re.search(r"\d+", user_input):
        response = f"Thank you. I will check your order status using this order/tracking ID: {user_input}"
        intent = "order_status_followup"
        confidence = 1.0
        model_used = "Context Memory"

        clear_context(user_id)

    # Context follow-up for return request
    elif last_intent == "return_request" and re.search(r"\d+", user_input):
        response = f"Thank you. I will start the return process for order ID: {user_input}"
        intent = "return_request_followup"
        confidence = 1.0
        model_used = "Context Memory"

        clear_context(user_id)

    # Context follow-up for cancellation confirmation
    elif last_intent == "cancellation" and user_input.lower().strip() in ["yes", "confirm", "sure", "yes confirm"]:
        response = "Your cancellation request has been confirmed and submitted."
        intent = "cancellation_confirmed"
        confidence = 1.0
        model_used = "Context Memory"

        clear_context(user_id)

    else:
        # Rule-based intent detection
        rule_intent, rule_confidence = rule_based_intent(user_input)

        if rule_intent is not None:
            intent = rule_intent
            confidence = rule_confidence
            model_used = "Rule Based"

        else:
            intent, confidence = predict_intent_transformer(user_input)
            model_used = "Pretrained Transformer"

        response = responses.get(
            intent,
            "I understand your message, but I need a little more detail to help you properly."
        )

        # Store context for intents that need follow-up
        if intent in ["order_status", "return_request", "cancellation"]:
            update_context(user_id, intent)
        else:
            clear_context(user_id)

    save_chat(
        user_id=user_id,
        user_message=user_input,
        bot_response=response,
        intent=intent,
        confidence=confidence,
        model_used=model_used
    )

    return {
        "response": response,
        "intent": intent,
        "confidence": round(float(confidence), 3),
        "model_used": model_used
    }

In [275]:
# ============================================================
# CELL 11: TEST CHATBOT
# ============================================================

test_queries = [
    "hi",
    "I forgot my password",
    "Where is my package?",
    "12345",
    "I want to cancel my subscription",
    "yes",
    "How can I update my payment method?",
    "I need technical support",
    "Tell me about your services",
    "What time do you open?"
]

for query in test_queries:
    result = chatbot_response(query)

    print("User:", query)
    print("Bot:", result["response"])
    print("Intent:", result["intent"])
    print("Confidence:", result["confidence"])
    print("Model Used:", result["model_used"])
    print("-" * 70)

User: hi
Bot: Hello! How can I help you today?
Intent: greeting
Confidence: 1.0
Model Used: Rule Based
----------------------------------------------------------------------
User: I forgot my password
Bot: You can reset your password using the 'Forgot Password' option. I can guide you step by step.
Intent: password_reset
Confidence: 1.0
Model Used: Rule Based
----------------------------------------------------------------------
User: Where is my package?
Bot: I can help you track your order. Please provide your order ID or tracking number.
Intent: order_status
Confidence: 1.0
Model Used: Rule Based
----------------------------------------------------------------------
User: 12345
Bot: Thank you. I will check your order status using this order/tracking ID: 12345
Intent: order_status_followup
Confidence: 1.0
Model Used: Context Memory
----------------------------------------------------------------------
User: I want to cancel my subscription
Bot: I can help you cancel your subscription

In [276]:
# ============================================================
# CELL 12: VIEW CHAT HISTORY
# ============================================================

history_df = load_chat_history(limit=10)
history_df

,user_message,bot_response,predicted_intent,confidence,model_used,timestamp
0,What time do you open?,Our support team is available from 9 AM to 6 P...,business_hours,1.0,Rule Based,2026-05-16T21:15:24.420186
1,Tell me about your services,"We provide account support, order tracking, re...",service_info,1.0,Rule Based,2026-05-16T21:15:24.411196
2,I need technical support,I can help with technical support. Please desc...,technical_support,1.0,Rule Based,2026-05-16T21:15:24.403190
3,How can I update my payment method?,You can update your payment method from your a...,payment_update,1.0,Rule Based,2026-05-16T21:15:24.395190
4,yes,Your cancellation request has been confirmed a...,cancellation_confirmed,1.0,Context Memory,2026-05-16T21:15:24.383221
5,I want to cancel my subscription,I can help you cancel your subscription. Pleas...,cancellation,1.0,Rule Based,2026-05-16T21:15:24.372752
6,12345,Thank you. I will check your order status usin...,order_status_followup,1.0,Context Memory,2026-05-16T21:15:24.360517
7,Where is my package?,I can help you track your order. Please provid...,order_status,1.0,Rule Based,2026-05-16T21:15:24.350997
8,I forgot my password,You can reset your password using the 'Forgot ...,password_reset,1.0,Rule Based,2026-05-16T21:15:24.342989
9,hi,Hello! How can I help you today?,greeting,1.0,Rule Based,2026-05-16T21:15:24.331450


In [277]:
# ============================================================
# CELL 13: INTERACTIVE CONSOLE CHATBOT
# ============================================================

def run_console_chatbot():
    """
    Run chatbot inside notebook or terminal.
    """

    print("Context-Aware Customer Support Chatbot Started")
    print("Type 'exit' to stop.")
    print("Type 'clear' to clear context.\n")

    user_id = "console_user"

    while True:
        user_input = input("You: ")

        if user_input.lower().strip() == "exit":
            print("Bot: Goodbye! Have a nice day.")
            break

        if user_input.lower().strip() == "clear":
            clear_context(user_id)
            print("Bot: Context cleared.")
            continue

        result = chatbot_response(user_input, user_id=user_id)

        print("Bot:", result["response"])
        print(f"Intent: {result['intent']} | Confidence: {result['confidence']} | Model: {result['model_used']}")
        print()


# Uncomment to run
# run_console_chatbot()

In [279]:
# ============================================================
# CELL 14: VOICE SEARCH / VOICE INPUT FUNCTIONALITY
# ============================================================

import speech_recognition as sr
import pyttsx3
import streamlit as st

# Initialize Text-to-Speech engine
tts_engine = pyttsx3.init()
tts_engine.setProperty("rate", 170)
tts_engine.setProperty("volume", 1.0)

def listen_to_user():
    """
    Capture voice input from microphone and convert to text.
    
    Returns:
        str: Recognized text or None if failed
    """
    recognizer = sr.Recognizer()
    
    try:
        with sr.Microphone() as source:
            print("🎤 Listening... Please speak.")
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=10)
        
        text = recognizer.recognize_google(audio)
        print(f"📝 Recognized: {text}")
        return text
    
    except sr.WaitTimeoutError:
        print("⏰ No speech detected within timeout")
        return None
    except sr.UnknownValueError:
        print("❌ Could not understand audio")
        return None
    except sr.RequestError as e:
        print(f"❌ Speech recognition service error: {e}")
        return None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def speak_response(text):
    """
    Convert chatbot text response into speech.
    
    Args:
        text: Text to speak
    """
    try:
        tts_engine.say(text)
        tts_engine.runAndWait()
    except Exception as e:
        print(f"❌ Text-to-speech error: {e}")

def voice_chat_session(user_id="voice_user"):
    """
    Run a complete voice-based chat session.
    
    Args:
        user_id: User identifier for context
    """
    print("\n" + "="*60)
    print("🎙️ VOICE CHAT MODE ACTIVATED")
    print("="*60)
    print("Commands:")
    print("  - Speak your query naturally")
    print("  - Say 'exit' to quit voice mode")
    print("  - Say 'clear' to clear context")
    print("="*60 + "\n")
    
    while True:
        user_input = listen_to_user()
        
        if user_input is None:
            print("Bot: I couldn't hear you. Please try again.\n")
            continue
        
        if user_input.lower() in ["exit", "quit", "goodbye"]:
            response = "Thank you for using voice chat. Goodbye!"
            print(f"Bot: {response}")
            speak_response(response)
            break
        
        if user_input.lower() == "clear":
            clear_context(user_id)
            response = "Conversation context has been cleared."
            print(f"Bot: {response}")
            speak_response(response)
            continue
        
        # Get chatbot response
        result = chatbot_response(user_input, user_id=user_id)
        response = result["response"]
        
        print(f"You: {user_input}")
        print(f"Bot: {response}")
        print(f"Intent: {result['intent']} | Confidence: {result['confidence']} | Model: {result['model_used']}\n")
        
        # Speak the response
        speak_response(response)

def voice_search(query=None):
    """
    Voice search function - speak your query and get response.
    For notebook usage.
    
    Args:
        query: Optional text query (if None, listens for voice)
    
    Returns:
        dict: Chatbot response
    """
    if query is None:
        query = listen_to_user()
    
    if query is None:
        return {"response": "No voice input detected.", "intent": "error", "confidence": 0.0, "model_used": "Voice"}
    
    result = chatbot_response(query, user_id="voice_search_user")
    
    print(f"\n🔍 Voice Search Query: {query}")
    print(f"🤖 Response: {result['response']}")
    print(f"📊 Intent: {result['intent']} | Confidence: {result['confidence']}")
    
    # Optional: Speak the response
    # speak_response(result['response'])
    
    return result

print("✅ Voice features loaded successfully!")
print("   - listen_to_user(): Capture voice input")
print("   - speak_response(): Convert text to speech")
print("   - voice_chat_session(): Run full voice chat")
print("   - voice_search(): Quick voice search")

✅ Voice features loaded successfully!
   - listen_to_user(): Capture voice input
   - speak_response(): Convert text to speech
   - voice_chat_session(): Run full voice chat
   - voice_search(): Quick voice search


In [283]:
# ============================================================
# CELL 15: VOICE CHAT DEMO
# ============================================================

def demo_voice_chat():
    """
    Simple demonstration of voice chat functionality.
    """
    print("\n" + "="*60)
    print("🎤 VOICE CHAT DEMO")
    print("="*60)
    print("This will listen to your voice and respond.")
    print("Say 'test' to try, or 'exit' to quit")
    print("="*60 + "\n")
    
    user_id = "demo_user"
    
    while True:
        print("Speak now...")
        user_input = listen_to_user()
        
        if user_input is None:
            print("No input detected. Try again.\n")
            continue
        
        if user_input.lower() == "exit":
            print("Exiting voice demo. Goodbye!")
            break
        
        if user_input.lower() == "test":
            print("🎤 Voice test successful! Say something else.\n")
            continue
        
        result = chatbot_response(user_input, user_id=user_id)
        
        print(f"\n🗣️ You said: {user_input}")
        print(f"🤖 Bot: {result['response']}")
        print(f"🎯 Intent: {result['intent']} (Confidence: {result['confidence']})\n")
        
        # Uncomment to hear response
        # speak_response(result['response'])

# Run voice demo
# demo_voice_chat()

In [288]:
# ============================================================
# CELL 16: VOICE-ENABLED CONSOLE CHAT (TEXT + VOICE)
# ============================================================

def run_voice_enabled_chat():
    """
    Run chatbot with both text and voice input options.
    """
    print("\n" + "="*60)
    print("🤖 VOICE-ENABLED CUSTOMER SUPPORT CHATBOT")
    print("="*60)
    print("Commands:")
    print("  - Type your message or say 'voice' to use voice input")
    print("  - Type 'exit' to quit")
    print("  - Type 'clear' to clear context")
    print("  - Type 'voice' to switch to voice mode for next message")
    print("="*60 + "\n")
    
    user_id = "console_user"
    voice_mode = False
    
    while True:
        if voice_mode:
            print("🎤 Voice mode active. Please speak...")
            user_input = listen_to_user()
            voice_mode = False
        else:
            user_input = input("You: ").strip()
        
        if user_input is None:
            print("Bot: No input detected. Please try again.\n")
            continue
        
        if user_input.lower() == "exit":
            print("Bot: Thank you for using the chatbot. Goodbye!")
            break
        
        if user_input.lower() == "clear":
            clear_context(user_id)
            print("Bot: Conversation context cleared.\n")
            continue
        
        if user_input.lower() == "voice":
            voice_mode = True
            print("🔄 Switching to voice mode for next message...\n")
            continue
        
        result = chatbot_response(user_input, user_id=user_id)
        
        print(f"Bot: {result['response']}")
        print(f"Intent: {result['intent']} | Confidence: {result['confidence']} | Model: {result['model_used']}\n")
        
        # Optional: Auto-speak responses
        # speak_response(result['response'])

# Run voice-enabled chat
# run_voice_enabled_chat()